In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv("PS_train.csv")
df = df.dropna()

In [ ]:
label2id = {label: i for i, label in enumerate(df["labels"].unique())}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df["labels"].map(label2id)

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["content"].tolist(),
    df["label_id"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label_id"]
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

In [ ]:
train_enc = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
class TamilDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = TamilDataset(train_enc, train_labels)
test_dataset = TamilDataset(test_enc, test_labels)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [ ]:
model.train()
epochs = 1

for epoch in range(epochs):
    total_loss = 0

    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 20 == 0:
            print(f"Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    print(f"Epoch Loss: {total_loss/len(train_loader):.4f}")

Batch 0/871 | Loss: 1.9449
Batch 20/871 | Loss: 1.9447
Batch 40/871 | Loss: 1.9381
Batch 60/871 | Loss: 1.9356
Batch 80/871 | Loss: 1.9282
Batch 100/871 | Loss: 1.9420
Batch 120/871 | Loss: 1.9030
Batch 140/871 | Loss: 1.9649
Batch 160/871 | Loss: 1.8573
Batch 180/871 | Loss: 1.9286
Batch 200/871 | Loss: 1.9179
Batch 220/871 | Loss: 1.9498
Batch 240/871 | Loss: 1.8637
Batch 260/871 | Loss: 1.8726
Batch 280/871 | Loss: 1.9118
Batch 300/871 | Loss: 1.8840
Batch 320/871 | Loss: 1.8962
Batch 340/871 | Loss: 1.8820
Batch 360/871 | Loss: 1.8714
Batch 380/871 | Loss: 1.8598
Batch 400/871 | Loss: 2.0032
Batch 420/871 | Loss: 1.8514
Batch 440/871 | Loss: 1.8867
Batch 460/871 | Loss: 1.8501
Batch 480/871 | Loss: 1.8457
Batch 500/871 | Loss: 1.8503
Batch 520/871 | Loss: 1.8563
Batch 540/871 | Loss: 1.8912
Batch 560/871 | Loss: 1.9562
Batch 580/871 | Loss: 1.8136
Batch 600/871 | Loss: 1.8394
Batch 620/871 | Loss: 1.9136
Batch 640/871 | Loss: 1.9056
Batch 660/871 | Loss: 1.8537
Batch 680/871 | Loss

In [ ]:
model.eval()
preds, true = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        true.extend(batch["labels"].numpy())

In [ ]:
print(classification_report(
    true,
    preds,
    target_names=label2id.keys(),
    zero_division=0
))

                   precision    recall  f1-score   support

          Neutral       0.00      0.00      0.00       128
    Substantiated       0.00      0.00      0.00        83
      Opinionated       0.31      1.00      0.48       272
         Positive       0.00      0.00      0.00       115
        Sarcastic       0.00      0.00      0.00       158
         Negative       0.00      0.00      0.00        81
None of the above       0.00      0.00      0.00        34

         accuracy                           0.31       871
        macro avg       0.04      0.14      0.07       871
     weighted avg       0.10      0.31      0.15       871



In [ ]:
 text = "இந்த படம் மிகவும் அருமை"

inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()
print("Prediction:", id2label[pred])

Prediction: Opinionated


In [ ]:
text = "இந்த படம் மிகவும் மோசமாக உள்ளது"

inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()
print("Prediction:", id2label[pred])

Prediction: Opinionated


In [ ]:
text = "இந்த படம் தமிழில் வெளியாகியுள்ளது"

inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()
print("Prediction:", id2label[pred])

Prediction: Opinionated
